<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-2-data-pipelines/Module_2_Session_3_API_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 2 — Session 3: API Data Pipelines

## What we are building
Swiggy doesn't just use its own database — it also pulls live data from
external APIs. For example: weather data to predict delivery delays,
or currency/location data for international expansion.

In this session we will:
1. Call a real public API
2. Parse the JSON response
3. Convert it into natural language
4. Feed it to an LLM for analysis

## Real-world equivalent
At Swiggy, this would be an API Gateway pulling external data,
processed by Lambda, and analyzed using Amazon Bedrock.

## Step 1 — Install and Import Libraries
New library this session: `requests`
- `requests` is the most popular Python library for calling APIs
- It lets us send a request to a URL and get back a response (usually JSON)

We also use `groq`, `os`, and `json` which are either familiar or built into Python.

In [6]:
# Install libraries
# requests is new this session — it lets us call any API on the internet
!pip install requests groq -q

In [7]:
# Import libraries
import requests  # lets us call APIs and get data back
import json  # built into Python — helps us read and pretty-print JSON
from groq import Groq  # to call the LLM
from google.colab import userdata  # to safely read Colab Secrets

## Step 2 — Call a Real Weather API
Swiggy uses weather data to predict delivery delays — heavy rain means
slower deliveries and more delivery partner safety concerns.

We use Open-Meteo — a free, no-API-key-needed weather API.
We will check the weather for Bangalore (a major Swiggy city).

In [8]:
# Open-Meteo API — free, no API key needed
# We give it Bangalore's latitude and longitude
url = "https://api.open-meteo.com/v1/forecast"

# Parameters for our API call
params = {
    "latitude": 12.9716,    # Bangalore latitude
    "longitude": 77.5946,   # Bangalore longitude
    "current_weather": True  # we only want current weather, not forecast
}

# Make the API call
response = requests.get(url, params=params)

# Check if the call was successful (status code 200 means OK)
print("Status Code:", response.status_code)

# Convert response to JSON (Python dictionary)
weather_data = response.json()

# Pretty print the full JSON response
print(json.dumps(weather_data, indent=2))

Status Code: 200
{
  "latitude": 12.970123,
  "longitude": 77.56364,
  "generationtime_ms": 0.1666545867919922,
  "utc_offset_seconds": 0,
  "timezone": "GMT",
  "timezone_abbreviation": "GMT",
  "elevation": 910.0,
  "current_weather_units": {
    "time": "iso8601",
    "interval": "seconds",
    "temperature": "\u00b0C",
    "windspeed": "km/h",
    "winddirection": "\u00b0",
    "is_day": "",
    "weathercode": "wmo code"
  },
  "current_weather": {
    "time": "2026-06-18T18:15",
    "interval": 900,
    "temperature": 21.4,
    "windspeed": 6.4,
    "winddirection": 223,
    "is_day": 0,
    "weathercode": 3
  }
}


In [9]:
weather_data["current_weather"]["temperature"]

21.4

## Step 3 — Extract Useful Fields
We don't need the entire JSON — just the fields relevant to delivery operations.
We pull out temperature, windspeed, and weathercode.

In [10]:
# Extract only the fields we care about for delivery operations
temperature = weather_data["current_weather"]["temperature"]
windspeed = weather_data["current_weather"]["windspeed"]
weathercode = weather_data["current_weather"]["weathercode"]

# Weather codes are numbers — we map common ones to readable text
# This is a simplified mapping (real WMO codes have more options)
weather_code_map = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Foggy",
    51: "Light drizzle",
    61: "Light rain",
    63: "Moderate rain",
    65: "Heavy rain",
    95: "Thunderstorm"
}

# Get readable weather description, default to "Unknown" if code not in our map
weather_description = weather_code_map.get(weathercode, "Unknown")

print(f"Temperature: {temperature}°C")
print(f"Windspeed: {windspeed} km/h")
print(f"Weather: {weather_description}")

Temperature: 21.4°C
Windspeed: 6.4 km/h
Weather: Overcast


## Step 4 — Convert to Natural Language and Send to LLM
We convert the weather data into a sentence, then ask the LLM to predict
delivery impact and suggest action for Swiggy operations team.

In [11]:
!pip install requests groq -q

In [12]:
# Re-run imports — Colab loses these if session restarts
import requests
import json
from groq import Groq
from google.colab import userdata

In [13]:
# Set up Groq client
client = Groq(api_key=userdata.get("GROQ_API_KEY"))

# Convert weather data into natural language
weather_text = (
    f"Current weather in Bangalore: {weather_description}, "
    f"temperature {temperature}°C, windspeed {windspeed} km/h."
)

print("Weather summary:", weather_text)
print()

# Build the prompt
prompt = f"""You are a Swiggy operations analyst.
Based on the current weather condition below, predict the impact on food delivery
and suggest one action the operations team should take.

Weather: {weather_text}

Give your response in this format:
Impact:
Suggested Action:
"""

# Send to LLM
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

print("LLM Response:")
print(response.choices[0].message.content)

Weather summary: Current weather in Bangalore: Overcast, temperature 21.4°C, windspeed 6.4 km/h.

LLM Response:
Impact:

 * Reduced demand for cold chain or ice-based delivery items due to cooler temperatures, potentially leading to increased delivery times for such food items
 * Possibly higher demand for hot beverages and comforting meals due to the overcast weather
 * Normal delivery times for most food items, as the temperatures are not significantly high or low

Suggested Action:
 
 * Monitor and adjust the 'Delivery Time' and 'Delivery Fee' for cold chain or ice-based delivery items to compensate for the potential increase in delivery time, maintaining customer satisfaction while taking into account operational constraints.


## What We Built Today
- Called a real live weather API using `requests`
- Parsed the JSON response and extracted useful fields
- Mapped weather codes to readable text
- Converted API data into natural language
- Sent it to an LLM for operational analysis

## AWS Equivalent
| What we did | AWS Service |
|---|---|
| Call external API | AWS API Gateway + Lambda |
| Parse and extract fields | AWS Lambda / Glue |
| Store raw API response | Amazon S3 |
| Send to LLM | Amazon Bedrock |
| Trigger on schedule | Amazon EventBridge |

## What is Next
Session 4 — Multimodal Pipelines
We will handle image inputs alongside text —
for example a Zepto product photo analyzed by an LLM.